[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/01_baseline_evaluation.ipynb)

# Step 1 — Baseline Evaluation

Build a held-out **policy-document test set** and measure how a small instruction-tuned model performs before synthetic-data alignment.

## Learning objectives
- Ingest two finance policy documents (policy-dense + scope-boundary)
- Split paragraphs into **test** vs **train** sets
- Generate hard test Q&A with a teacher LLM
- Run baseline inference and LLM-as-judge scoring by failure mode

![Model Improvement Loop Flowchart](./images/SLM_finetuning_flowchart.png)


## Setup

In [1]:
import os
from pathlib import Path

from aieng.syn_data.text import (
    BASELINE_PREDICTIONS_PATH,
    BASELINE_SCORES_PATH,
    DEFAULT_TEST_PARAS_PER_DOC,
    PARAGRAPHS_PATH,
    TEST_SET_PATH,
    ParagraphSplit,
    QASample,
    build_paragraph_splits,
    create_judge_client,
    create_small_model_client,
    create_teacher_client,
    generate_test_qa_batch,
    list_domain_documents,
    load_implementation_dotenv,
    run_inference,
    save_baseline_results,
    save_typed_jsonl,
    score_predictions,
)
from rich.console import Console
from rich.panel import Panel
from rich.syntax import Syntax
from rich.table import Table


# Setting the notebook directory to the project's root folder
if Path("").absolute().name == "synthetic-data-bootcamp":
    print(f"Notebook path is already the root path: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"The notebook path has been set to: {Path('').absolute()}")

load_implementation_dotenv()

console = Console(width=100)

The notebook path has been set to: /home/coder/synthetic-data-bootcamp


In [2]:
import logging


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,  # override any earlier basicConfig from other libs
)

## 1. Load finance policy documents

We use two document archetypes:
- **Policy-dense** (CFPB credit card agreement) → format + vocabulary + multi-constraint
- **Scope-boundary** (SEC investor bulletin) → refusal calibration

Two document types = two different skills the small model needs to learn.

Policy-dense (CFPB credit card agreement)

Lots of rules, numbers, fees, defined terms
Tests: format compliance (answer as JSON/table), domain vocabulary (APR, grace period), multi-constraint questions (“what’s the fee and when is it charged?”)
Scope-boundary (SEC investor bulletin)

Explains what the document covers — and what it doesn’t
Tests: refusal calibration — answer in-scope questions, politely refuse out-of-scope ones (e.g. “Should I buy this stock?”)

So the bootcamp uses two archetypes to build a test set and training data that stress different weaknesses — closer to real deployments where models handle both “answer precisely from policy” and “know their limits.”

In code, ``failure_modes_for_paragraph()`` maps each role to the failure modes it’s meant to target.

In [3]:
specs = list_domain_documents("finance")
from rich.console import Console


console = Console()
table = Table(show_header=True, header_style="bold magenta")
table.add_column("doc_id")
table.add_column("title")
table.add_column("role")
table.add_column("domain")
table.add_column("source_url")
table.add_column("local_path")

for spec in specs:
    table.add_row(
        getattr(spec, "doc_id", ""),
        getattr(spec, "title", ""),
        str(getattr(spec, "role", "")),
        getattr(spec, "domain", ""),
        getattr(spec, "source_url", ""),
        getattr(spec, "local_path", ""),
    )

console.print(table)

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ doc_id              ┃ title               ┃ role           ┃ domain  ┃ source_url         ┃ local_path          ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ cfpb_credit_card_a… │ CFPB Sample Credit  │ policy_dense   │ finance │ https://files.con… │ /home/coder/synthe… │
│                     │ Card Agreement      │                │         │                    │                     │
│ sec_investor_bulle… │ SEC Investor        │ scope_boundary │ finance │ https://www.sec.g… │ /home/coder/synthe… │
│                     │ Bulletin            │                │         │                    │                     │
└─────────────────────┴─────────────────────┴────────────────┴─────────┴────────────────────┴─────────────────────┘

## 2. Chunk into paragraphs and hold out test paragraphs

Randomly sample a few paragraphs per document for evaluation. **Never** use these paragraphs in Step 4 training.

In [5]:
paragraphs = build_paragraph_splits(
    "finance",
    n_test_per_doc=DEFAULT_TEST_PARAS_PER_DOC,
    seed=42,
)
test_paragraphs = [p for p in paragraphs if p.split == ParagraphSplit.TEST]
train_paragraphs = [p for p in paragraphs if p.split == ParagraphSplit.TRAIN]

console.print(f"[bold green]Total paragraphs:[/bold green] [cyan]{len(paragraphs)}[/cyan]")
console.print(
    f"[bold green]Test holdout:[/bold green] [cyan]{len(test_paragraphs)}[/cyan] | [bold green]Train reserve:[/bold green] [cyan]{len(train_paragraphs)}[/cyan]"
)

save_typed_jsonl(
    PARAGRAPHS_PATH,
    paragraphs,
    to_dict=lambda paragraph: paragraph.to_dict(),
)
PARAGRAPHS_PATH

Total paragraphs: 56

Test holdout: 20 | Train reserve: 36

PosixPath('/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/paragraphs.jsonl')

## 3. Generate hard test Q&A with the teacher model

Target the four small-model failure modes:
1. Format non-compliance
2. Domain vocabulary drift
3. Refusal vs engagement calibration
4. Multi-constraint collapse

## Why a scope-boundary doc?

You don’t need a second document for “synthetic QA from text” to work. You need it if you care about a specific failure mode: knowing when not to answer.

A policy-dense doc (credit-card agreement, HIPAA notice, tariff) trains the model to engage: quote rules, use terms, follow format, stack constraints. If that’s all you train on, the SLM learns “always produce a confident policy answer.”

A scope-boundary doc (SEC bulletin, “this is not medical advice,” “what we don’t cover”) trains the opposite skill: refuse or hedge when the question is outside the passage, out of product scope, or asking for advice the source doesn’t give.

Without that, your eval mostly measures “can it recite the handbook?” and will miss:

- Hallucinated answers when the passage doesn’t contain the fact
- Over-refusal on in-scope questions (after you punish hallucination)
- Mixing domains (“apply the credit-card fee rule to this SEC question”)

That’s why the pair exists: precision on in-scope policy vs calibration on out-of-scope asks. Same metrics, two skills.

If your product is closed-book FAQ over one handbook and you always retrieve a relevant chunk, you can drop the boundary doc and keep a negative test set instead: questions the retrieved text cannot answer, with gold “I don’t know / not in the document.” The boundary doc is just a convenient way to generate those cases.

In [6]:
teacher = create_teacher_client()

console.print(f"[bold magenta]Teacher LLM Base URL:[/bold magenta] [cyan]{teacher.settings.base_url}[/cyan]")

test_samples = generate_test_qa_batch(
    teacher,
    test_paragraphs,
    questions_per_para=3,
)
console.print(f"[bold green]Generated:[/bold green] [cyan]{len(test_samples)}[/cyan] test Q&A items")

save_typed_jsonl(
    TEST_SET_PATH,
    test_samples,
    to_dict=QASample.to_dict,
)

Teacher LLM Base URL: https://proxy.vectorinstitute.ai/v1

Generated: 56 test Q&A items

PosixPath('/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/test/test_set.jsonl')

Altrnatively, you may already saved the generated tests. So you can continue with reading them without generation:

In [7]:
from aieng.syn_data.text.io import load_typed_jsonl


test_samples = load_typed_jsonl(TEST_SET_PATH, QASample.from_dict)

In [8]:
def show_qa_samples(samples):
    """Display a table of Q&A samples with IDs, questions, answers, and metadata.

    Parameters
    ----------
    samples : list
        A list of QASample objects or similar, each with id, question, gold_answer,
        failure_mode, and role attributes.
    """
    table = Table(title="Test Q&A Samples", show_lines=True)
    table.add_column("ID", style="cyan", no_wrap=True)
    table.add_column("Question", style="magenta")
    table.add_column("Answer", style="green")
    table.add_column("Failure Mode", style="yellow")
    table.add_column("Role", style="blue")

    max_length = 200

    for sample in samples:
        table.add_row(
            sample.id,
            sample.question[:max_length] + "..." if len(sample.question) > max_length else sample.question,
            sample.gold_answer[:max_length] + "..." if len(sample.gold_answer) > max_length else sample.gold_answer,
            str(sample.failure_mode.value) if hasattr(sample.failure_mode, "value") else str(sample.failure_mode),
            str(sample.role.value) if hasattr(sample.role, "value") else str(sample.role),
        )

    console.print(table)

In [9]:
show_qa_samples(test_samples[25:27])

                                                 Test Q&A Samples                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ ID                                       ┃ Question         ┃ Answer          ┃ Failure Mode     ┃ Role         ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ test-cfpb_credit_card_agreement::p0031-0 │ What is the      │ Within 90 days  │ format_non_comp… │ policy_dense │
│                                          │ maximum time     │ of receiving    │                  │              │
│                                          │ frame within     │ your letter, we │                  │              │
│                                          │ which the        │ must either     │                  │              │
│                                          │ company must     │ correct the     │                  │              │
│                                          │ correct an error │ error or        │                  │              │
│                                          │ or provide an    │ explain to you  │                  │              │
│                                          │ explanation      │ why we believe  │                  │              │
│                                          │ after receiving  │ the bill is     │                  │              │
│                                          │ a letter about a │ correct.        │                  │              │
│                                          │ billing error?   │                 │                  │              │
├──────────────────────────────────────────┼──────────────────┼─────────────────┼──────────────────┼──────────────┤
│ test-cfpb_credit_card_agreement::p0031-1 │ What are the     │ During the      │ domain_vocabula… │ policy_dense │
│                                          │ obligations of   │ investigation   │                  │              │
│                                          │ the billing      │ of a billing    │                  │              │
│                                          │ entity during    │ dispute, the    │                  │              │
│                                          │ the              │ billing entity  │                  │              │
│                                          │ investigation of │ cannot try to   │                  │              │
│                                          │ a billing        │ collect the     │                  │              │
│                                          │ dispute, and how │ amount in       │                  │              │
│                                          │ might these      │ question or     │                  │              │
│                                          │ affect the       │ report the      │                  │              │
│                                          │ consumer's       │ consumer as     │                  │              │
│                                          │ credit limit?    │ delinquent on   │                  │              │
│                                          │                  │ that amount.    │                  │              │
│                                          │                  │ However, the    │                  │              │
│                                          │                  │ charge may      │                  │              │
│                                          │                  │ remain on th... │                  │              │
└──────────────────────────────────────────┴──────────────────┴─────────────────┴──────────────────┴──────────────┘

## 4. Baseline inference with the small model

Plug in your small model client here (local GGUF, Ollama, or HF 4-bit model).

In [19]:
small_model = create_small_model_client()

predictions = run_inference(small_model, test_samples)
console.print(f"[bold green]Collected:[/bold green] [cyan]{len(predictions)}[/cyan] baseline predictions")


sample = predictions[0]
sample_dict = sample.to_dict() if hasattr(sample, "to_dict") else dict(sample)
pretty_json = Syntax.from_json(sample_dict, indent=2) if hasattr(Syntax, "from_json") else None

if pretty_json is not None:
    console.print(Panel(pretty_json, title="First Prediction"))
else:
    # Fallback if Syntax.from_json is not available
    import json

    formatted = json.dumps(sample_dict, indent=2)
    console.print(Panel(formatted, title="First Prediction"))

2026-08-27 19:54:40,030 INFO aieng.syn_data.text.small_model: Creating small model client for model: qwen2.5:0.5b-instruct


qwen2.5:0.5b-instruct created


Collected: 56 baseline predictions

╭─────────────────────────────────────────────── First Prediction ────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "id": "test-cfpb_credit_card_agreement::p0005-0",                                                             │
│   "question": "Under what conditions will new purchases posted to your account during a billing cycle incur a   │
│ finance charge?",                                                                                               │
│   "gold_answer": "New purchases will incur a finance charge if you did not have a zero or credit balance at the │
│ beginning of the billing cycle and did not pay the entire new balance on the previous cycle's billing statement │
│ by the payment due date. In such cases, a finance charge will accrue from the date a purchase is posted to your │
│ account.",                                                                                                      │
│   "model_answer": "Under the given context, new purchases posted to your account during a billing cycle will    │
│ not incur a finance charge if:\n\n1. You had a zero or credit balance at the beginning of that billing          │
│ cycle.\n2. You paid the entire new balance on the previous cycle's billing statement by the payment due         │
│ date.\n\nThese conditions ensure that the finance charge is calculated separately for purchases and cash        │
│ advances, avoiding additional charges based on the same transactions.",                                         │
│   "failure_mode": "format_non_compliance",                                                                      │
│   "doc_id": "cfpb_credit_card_agreement",                                                                       │
│   "para_id": "cfpb_credit_card_agreement::p0005"                                                                │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 5. LLM-as-judge baseline scores

In [20]:
judge = create_judge_client()

baseline_scores = score_predictions(judge, test_samples, predictions)
baseline_summary = save_baseline_results(
    predictions,
    baseline_scores,
    test_samples,
    predictions_path=BASELINE_PREDICTIONS_PATH,
    scores_path=BASELINE_SCORES_PATH,
)

# Baseline score for one sample
sample_score = baseline_scores[0]
score_dict = sample_score.to_dict() if hasattr(sample_score, "to_dict") else dict(sample_score)
score_dict["average"] = getattr(sample_score, "average", None)
pretty_json = Syntax.from_json(score_dict, indent=2) if hasattr(Syntax, "from_json") else None

if pretty_json is not None:
    console.print(Panel(pretty_json, title="Sample Baseline Score"))
else:
    import json

    formatted = json.dumps(score_dict, indent=2)
    console.print(Panel(formatted, title="Sample Baseline Score"))
Path("implementations/qa_text_generation/slm_baseline_before.md").write_text(str(sample_score) + "\n")

2026-08-27 19:56:11,249 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0005-0 (answer length: 466)
2026-08-27 19:56:12,949 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0005-1 (answer length: 346)
2026-08-27 19:56:14,023 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0005-2 (answer length: 611)
2026-08-27 19:56:15,412 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0009-0 (answer length: 533)
2026-08-27 19:56:16,758 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0009-1 (answer length: 501)
2026-08-27 19:56:18,726 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0009-2 (answer length: 782)
2026-08-27 19:56:19,856 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-

╭───────────────────────────────────────────── Sample Baseline Score ─────────────────────────────────────────────╮
│ {                                                                                                               │
│   "sample_id": "test-cfpb_credit_card_agreement::p0005-0",                                                      │
│   "correctness": 3.0,                                                                                           │
│   "coherence": 5.0,                                                                                             │
│   "instruction_following": 4.0,                                                                                 │
│   "factual_plausibility": 5.0,                                                                                  │
│   "reasoning": "The model answered the inverse of the question (stating when charges will NOT be incurred       │
│ instead of when they WILL be incurred), although the underlying logic is correct based on the reference.",      │
│   "metadata": {},                                                                                               │
│   "average": 4.25                                                                                               │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Baseline score summary

In [21]:
# Create a table to display the baseline_summary
table = Table(title="Baseline Summary", highlight=True)

# Define the columns based on the baseline_summary structure
table.add_column("Failure Mode", style="bold cyan")
table.add_column("Correctness", justify="right", style="green")
table.add_column("Coherence", justify="right", style="green")
table.add_column("Instruction Following", justify="right", style="green")
table.add_column("Factual Plausibility", justify="right", style="green")
table.add_column("Average", justify="right", style="bold yellow")

# Add the "overall" scores as the first row
overall = baseline_summary.get("overall", {})
table.add_row(
    "[b]Overall[/b]",
    f"{overall.get('correctness', 0):.2f}",
    f"{overall.get('coherence', 0):.2f}",
    f"{overall.get('instruction_following', 0):.2f}",
    f"{overall.get('factual_plausibility', 0):.2f}",
    f"{overall.get('average', 0):.2f}",
)

# Add a row for each failure mode
by_failure = baseline_summary.get("by_failure_mode", {})
for mode, scores in by_failure.items():
    table.add_row(
        mode,
        f"{scores.get('correctness', 0):.2f}",
        f"{scores.get('coherence', 0):.2f}",
        f"{scores.get('instruction_following', 0):.2f}",
        f"{scores.get('factual_plausibility', 0):.2f}",
        f"{scores.get('average', 0):.2f}",
    )

console.print(table)

                                                Baseline Summary                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Failure Mode              ┃ Correctness ┃ Coherence ┃ Instruction Following ┃ Factual Plausibility ┃ Average ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ Overall                   │        3.91 │      4.75 │                  4.75 │                 4.16 │    4.39 │
│ format_non_compliance     │        4.45 │      4.89 │                  4.95 │                 4.61 │    4.72 │
│ domain_vocabulary_drift   │        3.83 │      4.67 │                  4.78 │                 3.89 │    4.29 │
│ multi_constraint_collapse │        2.56 │      4.22 │                  4.44 │                 2.78 │    3.50 │
│ refusal_calibration       │        4.05 │      4.89 │                  4.68 │                 4.50 │    4.53 │
└───────────────────────────┴─────────────┴───────────┴───────────────────────┴──────────────────────┴─────────┘